# Classification de texte : un classifieur Naive Bayes from scratch

**Série NLP — carnet 10.** Prolonge [02 · N-grammes](02_NGrammes_Modeles_De_Langue.ipynb) (estimation par MLE + lissage) et [06 · Recherche d'information](06-Recherche-Information-TFIDF-BM25.ipynb) (le sac-de-mots).

**Navigation** : [<< Précédent — 09. fastText (subwords)](09-FastText-Subword-Vectors.ipynb) | [Index de la série](README.md) | **10 (ce notebook)** | [Suivant — 11. Régression logistique >>](11-Classement-Texte-RegressionLogistique-FromScratch.ipynb)

Les carnets précédents demandent « quelle est la probabilité de ce texte ? » (02) ou « quel document est le plus pertinent pour cette requête ? » (06). Ici on demande « **quelle est la catégorie de ce document ?** ». Le **Naive Bayes** répond en factorisant la probabilité d'un document par sa catégorie grâce à l'hypothèse d'indépendance « naïve » des mots :

$$P(c \mid d) \;\propto\; P(c)\prod_{w\in d} P(w \mid c)$$

Tout est **mesuré** (précision out-of-sample sur un corpus auto-contenu, aucun téléchargement) et le scoreur from scratch est **confronté à `sklearn`** — la machine réelle — sur le même split, avec le même vocabulaire d'apprentissage et la même tokenisation.

**Version jumelle** : [Python](10-Classement-Texte-NaiveBayes-FromScratch.ipynb).

**Objectifs** : construire et évaluer Naive Bayes, interpréter le lissage et comparer à scikit-learn.  
**Prérequis** : C#, probabilités et dictionnaires. **Durée** : 60 minutes.

**Interopérabilité** : installer CPython avec scikit-learn et NumPy dans l'environnement Python, puis définir `PYTHONNET_PYDLL` vers la bibliothèque partagée Python. Définir au besoin `PYTHONHOME` et `PYTHONPATH` avant de démarrer Jupyter pour sélectionner le Python et ses paquets ; les valeurs dépendent de votre installation. Le calcul principal est en C# ; Python.NET sert seulement au témoin scikit-learn.


In [1]:
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text.RegularExpressions;
using System.Globalization;
using Python.Runtime;
var corpus = new Dictionary<string,string[]> {
    ["sport"] = new [] { "l'equipe du stade remporte le match sur un dernier but", "l'arbitre siffle la fin du match dans le stade", "le coach motive l'equipe avant le match decisif", "le gardien arrete le penalty au dernier moment", "les joueurs celebrent la victoire dans le stade", "le buteur marque le but decisif du match", "le score final du match est historique au stade", "l'equipe entree au stade pour le match de la saison" },
    ["tech"] = new [] { "le developp corrige un bug dans le logiciel", "le serveur renvoie une erreur dans le log du systeme", "la puce integre un accelerateur dans le processeur", "le framework compile le code en quelques secondes", "l'application consomme la memoire au demarrage du systeme", "l'algorithme accelere le calcul dans le logiciel", "l'interface du logiciel interroge le serveur du systeme", "le code du systeme compile sans erreur" },
    ["science"] = new [] { "l'experience mesure la constante avec une grande precision", "le paper publie les resultats de l'observation", "l'hypothese est confirmee par les donnees recueillies", "le chercheur analyse les echantillons au microscope", "la theorie predit le comportement du systeme physique", "l'article rapporte une mesure de l'energie", "les mesures de l'experience valident le modele theorique", "l'observation confirme le resultat de l'experience" },
    ["finance"] = new [] { "la bourse enregistre une hausse des indices ce matin", "la banque annonce un resultat net en hausse", "l'action cote plus haut a la fin de la seance", "l'investisseur choisit l'obligation pour le credit", "le taux d'interet guide la decision d'emprunt", "le portefeuille subit une baisse des valeurs", "le dividende est verse aux actionnaires du marche", "le marche anticipe une revision des taux" }
};
var categories = corpus.Keys.ToArray();
Console.WriteLine($".NET {Environment.Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");
Console.WriteLine($"Corpus : {corpus.Sum(x => x.Value.Length)} documents, {categories.Length} catégories");

Installed Packages pythonnet, 3.0.5

.NET 10.0.14 | Python.NET 3.0.5.0


Corpus : 32 documents, 4 catégories


## 1. Le sac-de-mots — tokenisation from scratch

Comme le carnet 06, on réduit chaque document à un **sac de mots** (multiset). Tokenisation minimale : minuscules, mots = suites d'alphabétiques. Aucun outil externe.

In [2]:
string[] Tokeniser(string texte) => Regex.Matches(texte.ToLowerInvariant(), "[a-z]+").Select(m => m.Value).ToArray();
var train = corpus.ToDictionary(kv => kv.Key, kv => kv.Value.Take(6).ToArray());
var test = corpus.ToDictionary(kv => kv.Key, kv => kv.Value.Skip(6).ToArray());
var vocab = train.Values.SelectMany(x => x).SelectMany(Tokeniser).Distinct().OrderBy(x => x, StringComparer.Ordinal).ToArray();
Console.WriteLine($"Vocabulaire du train : {vocab.Length} mots | exemples : {string.Join(", ", vocab.Take(12))}");

Vocabulaire du train : 118 mots | exemples : a, accelerateur, accelere, action, algorithme, analyse, annonce, application, arbitre, arrete, article, au


## 2. L'estimateur Naive Bayes — MLE + lissage, en log-espace

L'hypothèse « naïve » factorise la vraisemblance. On estime par **maximum de vraisemblance** (comptages) avec **lissage de Laplace (add-α)** pour qu'aucun mot absent du train ne fasse s'effondrer le produit à 0 :

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\text{count}(\ast, c) + \alpha\,|V|}$$

Le scoreur somme les **log-probabilités** (stabilité numérique) : $\log P(c) + \sum_w \log P(w \mid c)$. C'est exactement la tradition du carnet 02 (MLE + lissage), appliquée aux catégories au lieu des n-grammes.

In [3]:
(Dictionary<string,double>, Dictionary<string,Dictionary<string,double>>) Entrainer(Dictionary<string,string[]> donnees, string[] mots, double alpha = 1.0) {
    var priors = new Dictionary<string,double>(); var probs = new Dictionary<string,Dictionary<string,double>>();
    int n = donnees.Values.Sum(x => x.Length);
    foreach (var (c, docs) in donnees) {
        var tokens = docs.SelectMany(Tokeniser).ToArray();
        var counts = tokens.GroupBy(x => x).ToDictionary(g => g.Key, g => g.Count());
        priors[c] = Math.Log((double)docs.Length / n);
        probs[c] = mots.ToDictionary(w => w, w => counts.GetValueOrDefault(w) + alpha > 0
            ? Math.Log((counts.GetValueOrDefault(w) + alpha) / (tokens.Length + alpha * mots.Length))
            : double.NegativeInfinity);
    }
    return (priors, probs);
}
Dictionary<string,double> Scorer(string doc, Dictionary<string,double> priors,
    Dictionary<string,Dictionary<string,double>> probs) =>
    priors.ToDictionary(kv => kv.Key, kv => kv.Value + Tokeniser(doc)
        .Where(w => probs[kv.Key].ContainsKey(w)).Sum(w => probs[kv.Key][w]));
(string, Dictionary<string,double>) Predire(string doc, Dictionary<string,double> priors, Dictionary<string,Dictionary<string,double>> probs) {
    var scores = Scorer(doc, priors, probs); var finis = scores.Where(kv => double.IsFinite(kv.Value)).OrderByDescending(kv => kv.Value).ToArray();
    return (finis.Length == 0 ? null : finis[0].Key, scores);
}
var (logPriors, logProbs) = Entrainer(train, vocab);
Console.WriteLine($"Train : {train.Values.Sum(x => x.Length)} | test : {test.Values.Sum(x => x.Length)}");

Train : 24 | test : 8


Le modèle est entraîné sur 24 documents. Évaluons-le maintenant sur les huit documents réservés au test.

In [4]:
var essais = categories.SelectMany(c => test[c].Select(t => (Doc:t, Vrai:c))).ToArray();
foreach (var e in essais) Console.WriteLine($"vrai={e.Vrai,-8} pred={Predire(e.Doc, logPriors, logProbs).Item1,-8} {e.Doc}");
Console.WriteLine($"Précision : {essais.Count(e => Predire(e.Doc, logPriors, logProbs).Item1 == e.Vrai)}/{essais.Length}");
var diagnostic = essais.Single(e => e.Doc.StartsWith("le dividende"));
Console.WriteLine("OOV : " + string.Join(", ", Tokeniser(diagnostic.Doc).Where(w => !vocab.Contains(w))));
foreach (var w in new[]{"le","est","du"}) Console.WriteLine($"{w}: " + string.Join(", ", categories.Select(c => $"{c}={train[c].SelectMany(Tokeniser).Count(t => t == w)}")));
var second = essais.Last(); Console.WriteLine("Deuxième finance — OOV : " + string.Join(", ", Tokeniser(second.Doc).Where(w => !vocab.Contains(w))));

vrai=sport    pred=sport    le score final du match est historique au stade


vrai=sport    pred=sport    l'equipe entree au stade pour le match de la saison


vrai=tech     pred=tech     l'interface du logiciel interroge le serveur du systeme


vrai=tech     pred=tech     le code du systeme compile sans erreur


vrai=science  pred=science  les mesures de l'experience valident le modele theorique


vrai=science  pred=science  l'observation confirme le resultat de l'experience


... (4 outputs left out)

le: sport=9, tech=9, science=3, finance=3


est: sport=0, tech=0, science=1, finance=0


du: sport=3, tech=2, science=1, finance=0


Deuxième finance — OOV : marche, anticipe, revision


### Lecture du résultat

Sept documents sur huit sont bien classés. L'erreur n'est pas un défaut d'implémentation : c'est ce que le modèle **pouvait** voir.

Le document mal classé, « le dividende est verse aux actionnaires du marche », se découpe en huit mots dont **cinq sont hors vocabulaire** — `dividende`, `verse`, `aux`, `actionnaires`, `marche`. Trois d'entre eux portent le sujet financier (`dividende`, `actionnaires`, `marche`) ; `verse` est le verbe et `aux` une préposition, absents du train par rareté et non par topicalité. Il ne reste au scoreur que trois mots connus du train :

| mot | sport | tech | science | finance |
|---|---|---|---|---|
| `le` | 9 | 9 | 3 | 3 |
| `est` | 0 | 0 | 1 | 0 |
| `du` | 3 | 2 | 1 | 0 |

Aucun ne penche vers `finance` : deux sur trois la contredisent, le troisième est absent de sa classe. Le classifieur a donc été privé de toute preuve utile, et la taille du vocabulaire du train — 118 mots pour 24 documents — dit pourquoi. `Scorer` n'additionne que les mots vus au train : ignorer les mots inconnus est nécessaire pour ne pas planter, mais transforme ici une absence d'information en information neutre.

Le second document finance est classé correctement alors qu'il perd lui aussi des mots — `marche`, `anticipe` et `revision` sont hors vocabulaire. Sa bonne classification ne doit donc rien au mot qui nomme le sujet : elle tient à `taux`, le seul mot financier que le train connaisse. Une lecture qui attribuerait le succès à `marche` se tromperait de preuve, exactement comme la précédente.

C'est la leçon transposable : une erreur de prédiction se diagnostique en regardant **quels mots ont réellement été comptés**, pas en soupçonnant le calcul. Un compte de comptages se lit avant un compte de scores.

### Exercice 1

Construisez les probabilités de présence Bernoulli. La fixture comporte des répétitions et un mot absent ; au scoring, les absences contribuent aussi.

In [5]:
// Exercice 1 : Bernoulli, probabilités de présence par classe.
Dictionary<string,Dictionary<string,double>> Bernoulli(Dictionary<string,string[]> docs, string[] mots, double alpha) {
    // TODO : p = (df + alpha) / (nombre de documents de la classe + 2 * alpha).
    // Au scoring inclure log(p) si présent et log(1-p) si absent.
    return null;
}
var exercice1 = Bernoulli(new Dictionary<string,string[]> {
    {"c", new[]{"x x y", "y"}} }, new[]{"x", "y", "z"}, 1.0);
if (exercice1 == null) {
    Console.WriteLine("Exercice 1 à compléter");
} else {
    var attendus = new Dictionary<string,double>{{"x",0.5},{"y",0.75},{"z",0.25}};
    var autre = Bernoulli(new Dictionary<string,string[]> {
        {"d",new[]{"x", "x", "y"}} }, new[]{"x","y","z"}, 1.0);
    var attendus2 = new Dictionary<string,double>{{"x",0.6},{"y",0.4},{"z",0.2}};
    bool Valider(Dictionary<string,Dictionary<string,double>> reponse, string cle,
                 Dictionary<string,double> valeurs) =>
        reponse != null && reponse.Count == 1 && reponse.ContainsKey(cle) &&
        reponse[cle].Count == valeurs.Count && valeurs.All(kv =>
            reponse[cle].TryGetValue(kv.Key, out var v) && double.IsFinite(v) &&
            Math.Abs(v - kv.Value) < 1e-9);
    Console.WriteLine(Valider(exercice1, "c", attendus) && Valider(autre, "d", attendus2)
        ? "Fixtures Bernoulli validées" : "Réponse incorrecte");
}

Exercice 1 à compléter


## 3. Cross-check SOTA — la machine from scratch contre `sklearn`

Le scoreur **from scratch** est confronté à `CountVectorizer` + `MultinomialNB` sur le même split. Le vocabulaire est construit sur le train seul, les mots OOV sont ignorés, et le témoin reçoit `token_pattern=r"[a-z]+"` : il conserve ainsi les mots d'une lettre, exactement comme `tokeniser`. Les différences de conventions sont donc explicitement contrôlées.


In [6]:
// Le Python ne reçoit que les données du corpus et les paramètres du témoin.
if (!PythonEngine.IsInitialized)
{
    var pythonDll = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (string.IsNullOrWhiteSpace(pythonDll))
        throw new InvalidOperationException("Définissez PYTHONNET_PYDLL avant le démarrage de Jupyter.");
    Runtime.PythonDLL = pythonDll;
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}
var xtrain = categories.SelectMany(c => train[c]).ToArray();
var ytrain = categories.SelectMany(c => train[c].Select(_ => c)).ToArray();
var xtest = essais.Select(e => e.Doc).ToArray();
string[] skPred;
using (Py.GIL()) {
    dynamic sys = Py.Import("sys"); dynamic sklearn = Py.Import("sklearn");
    dynamic feature = Py.Import("sklearn.feature_extraction.text"); dynamic naive = Py.Import("sklearn.naive_bayes");
    dynamic vec = feature.CountVectorizer(token_pattern: "[a-z]+");
    dynamic matrix = vec.fit_transform(xtrain.ToPython());
    dynamic clf = naive.MultinomialNB(alpha: 1.0); clf.fit(matrix, ytrain.ToPython());
    using PyObject predictions = clf.predict(vec.transform(xtest.ToPython()));
    skPred = predictions.InvokeMethod("tolist").As<string[]>();
    Console.WriteLine($"Python {sys.version} | scikit-learn {sklearn.__version__}");
}
var fsPred = essais.Select(e => Predire(e.Doc, logPriors, logProbs).Item1).ToArray();
Console.WriteLine($"Précision C# {essais.Where((e,i) => fsPred[i] == e.Vrai).Count()}/8 | sklearn {essais.Where((e,i) => skPred[i] == e.Vrai).Count()}/8 | accord {fsPred.Where((x,i) => x == skPred[i]).Count()}/8");
for(int i=0;i<essais.Length;i++) Console.WriteLine($"{essais[i].Vrai,-8} C#={fsPred[i],-8} sklearn={skPred[i],-8} {essais[i].Doc}");

Python 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | scikit-learn 1.8.0


Précision C# 7/8 | sklearn 7/8 | accord 8/8


sport    C#=sport    sklearn=sport    le score final du match est historique au stade


sport    C#=sport    sklearn=sport    l'equipe entree au stade pour le match de la saison


tech     C#=tech     sklearn=tech     l'interface du logiciel interroge le serveur du systeme


tech     C#=tech     sklearn=tech     le code du systeme compile sans erreur


science  C#=science  sklearn=science  les mesures de l'experience valident le modele theorique


science  C#=science  sklearn=science  l'observation confirme le resultat de l'experience


finance  C#=sport    sklearn=sport    le dividende est verse aux actionnaires du marche


finance  C#=finance  sklearn=finance  le marche anticipe une revision des taux


### Lecture du résultat

Deux implémentations écrites indépendamment donnent le **même compte** (7/8) et, plus parlant, le **même verdict sur chacun des huit documents** — y compris le même désaccord avec la vérité (`le dividende ...` classé `sport` des deux côtés).

Il faut lire ce que cet accord prouve et ce qu'il ne prouve pas. Il prouve que la machine from scratch **calcule ce qu'elle prétend calculer** : même tokenisation, même vocabulaire construit sur le train, même ignorance des mots hors vocabulaire, même estimateur. C'est une validation de **code**.

Il ne prouve rien sur le **modèle**. Un témoin qui reproduit une erreur la confirme au lieu de l'absoudre : ici le désaccord avec la vérité n'est pas un bruit d'implémentation qu'un meilleur code effacerait, c'est la même absence d'information qui frappe les deux, parce qu'elle est dans les données — un vocabulaire de train trop petit pour couvrir le document — et non dans l'algorithme.

Un cross-check SOTA répond donc à « est-ce que j'ai bien codé ? », jamais à « est-ce que le modèle est bon ? ». La seconde question demande un autre instrument : plus de données, ou une mesure hors de ce test.

### Exercice 2

Expliquez un score en conservant les occurrences connues. Le mot hors vocabulaire est ignoré, comme par le scoreur.

In [7]:
// Exercice 2 : contributions connues ; les mots OOV sont ignorés.
List<(string Mot,double Valeur)> Contributions(string doc, string classe, Dictionary<string,Dictionary<string,double>> probs) {
    // TODO : conserver les occurrences connues, trier par contribution décroissante.
    return null;
}
var exercice2 = Contributions("le match inconnu match", "sport", logProbs);
if (exercice2 == null) {
    Console.WriteLine("Exercice 2 à compléter");
} else {
    var attendus = new[]{("le", logProbs["sport"]["le"]),
                         ("match", logProbs["sport"]["match"]),
                         ("match", logProbs["sport"]["match"])};
    var autre = Contributions("serveur inconnu", "tech", logProbs);
    bool valide = exercice2.Count == attendus.Length &&
        exercice2.Zip(attendus, (x, a) => x.Mot == a.Item1 &&
            double.IsFinite(x.Valeur) && Math.Abs(x.Valeur - a.Item2) < 1e-9).All(x => x) &&
        autre != null && autre.Count == 1 && autre[0].Mot == "serveur" &&
        double.IsFinite(autre[0].Valeur) &&
        Math.Abs(autre[0].Valeur - logProbs["tech"]["serveur"]) < 1e-9;
    Console.WriteLine(valide ? "Fixtures contributions validées" : "Réponse incorrecte");
}

Exercice 2 à compléter


## 4. Ce que le lissage change — l'effet de α, mesuré

Sans lissage (`α = 0`), un mot connu du vocabulaire mais absent d'une catégorie rend son score `-inf`. Si toutes les catégories sont à `-inf`, le document est **indéterminé**, pas classé par l'ordre du dictionnaire. Pour α > 0, la table donne la précision sur 8 documents : sa résolution est donc de 12,5 points, insuffisante pour distinguer un vrai plateau entre plusieurs valeurs proches.


In [8]:
foreach(double alpha in new[]{0.0,0.1,0.5,1.0,2.0,5.0}) {
    var (lp,lw) = Entrainer(train,vocab,alpha);
    var results = essais.Select(e => Predire(e.Doc,lp,lw)).ToArray();
    var margins = results.Select(r => r.Item2.Values.Where(double.IsFinite)
        .OrderByDescending(x => x).ToArray()).Where(x => x.Length >= 2)
        .Select(x => x[0] - x[1]).OrderBy(x => x).ToArray();
    string median = margins.Length == 0 ? "-" :
        ((margins[(margins.Length-1)/2] + margins[margins.Length/2])/2)
        .ToString("F2", CultureInfo.InvariantCulture);
    int indetermines = results.Count(r => r.Item1 == null);
    int classes = results.Length - indetermines;
    Console.WriteLine($"alpha={alpha.ToString(CultureInfo.InvariantCulture),3} indéterminés={indetermines} classes={classes} marge médiane={median,5} précision={results.Where((r,i)=>r.Item1==essais[i].Vrai).Count()}/8");
}

alpha=  0 indéterminés=3 classes=5 marge médiane=    - précision=4/8


alpha=0.1 indéterminés=0 classes=8 marge médiane= 6.33 précision=7/8


alpha=0.5 indéterminés=0 classes=8 marge médiane= 3.39 précision=7/8


alpha=  1 indéterminés=0 classes=8 marge médiane= 2.15 précision=7/8


alpha=  2 indéterminés=0 classes=8 marge médiane= 1.36 précision=7/8


alpha=  5 indéterminés=0 classes=8 marge médiane= 0.59 précision=7/8


### Lecture du résultat

`α = 0` classe 5 documents sur 8 — 4 justes et 1 faux — et déclare les 3 autres **indéterminés** — pas « mal classés », indéterminés : aucune catégorie n'échappe au `-inf`. Il suffit d'un seul mot du document absent d'une classe pour la couler, puisque la somme de log-probabilités est contaminée par un seul `-inf`. Les trois documents qui tombent sont les deux `sport` et un `science`, ceux dont chaque catégorie ignore au moins un mot.

La colonne `marge mediane` affiche `-` à `α = 0`, et ce n'est pas une case manquante : aucun des cinq documents classés n'y garde deux scores finis, donc il n'y a aucun écart à mesurer. La décision n'y est plus un arbitrage entre catégories, mais la survie d'une seule — ce que le compteur `classes` sépare de la précision, qui ne dit que le nombre de bonnes réponses.

Puis la précision vaut 7/8 pour **toutes** les valeurs testées de 0,1 à 5,0. Il serait tentant d'en conclure que `α` n'a pas d'importance. La marge de décision dit le contraire :

| α | écart médian entre la 1re et la 2e catégorie |
|---|---|
| 0,1 | 6,33 |
| 1,0 | 2,15 |
| 5,0 | 0,59 |

La précision est stable, la **confiance ne l'est pas** : elle est divisée par onze. À `α = 5`, le terme de lissage `α·|V| = 590` pèse plus de dix fois les jetons réellement observés dans une catégorie (48 à 54), et la marge médiane tombe sous 0,6 nat — un mot de plus ou de moins peut renverser le verdict.

Le plateau mesure donc la **granularité d'un test de huit documents** (douze points et demi par document), pas l'insensibilité du modèle à `α`. Deux régimes différents — l'évidence des mots à petit `α`, le dénominateur à grand `α` — peuvent rendre le même compte. Un chiffre de précision ne se lit jamais seul ; la marge dit ce qu'il coûte.

### Exercice 3

Mesurez la précision après filtrage et comparez deux jeux de mots vides : aucun mot et tout le vocabulaire du train.

In [9]:
// Exercice 3 : mesurer la précision après filtrage des mots vides.
double? PrecisionSansMotsVides(HashSet<string> vides) {
    // TODO : filtrer train et test, reconstruire le vocabulaire, entraîner et mesurer.
    return null;
}
var exercice3 = PrecisionSansMotsVides(new HashSet<string>());
if (exercice3 == null) Console.WriteLine("Exercice 3 à compléter");
else {
    var tout = PrecisionSansMotsVides(vocab.ToHashSet());
    bool valide = double.IsFinite(exercice3.Value) && Math.Abs(exercice3.Value - 0.875) < 1e-9 &&
        tout.HasValue && double.IsFinite(tout.Value) && Math.Abs(tout.Value - 0.25) < 1e-9;
    Console.WriteLine(valide ? "Fixtures précision validées" : "Réponse incorrecte");
}

Exercice 3 à compléter


## 5. Synthèse

| Question | Carnet | Machine |
|---|---|---|
| Probabilité du texte ? | 02 | n-grammes (MLE + lissage) |
| Document le plus pertinent ? | 06 | BM25 / TF-IDF |
| **Catégorie du document ? (ici)** | **10** | **Naive Bayes (MLE + lissage)** |

Le Naive Bayes from scratch **atteint une précision OOS mesurée** et **coïncide avec `sklearn`** sur ce petit test. Les deux machines partagent ici le vocabulaire du train, l'ignorance des mots OOV et la tokenisation ; le carnet 02 donne la même leçon d'estimation pour les n-grammes. La précision est mesurée sur 8 documents, pas une estimation de performance générale.


## 6. Exercices : récapitulatif

Les trois exercices répartis dans le carnet portent sur [Bernoulli](#exercice-1), les [contributions](#exercice-2) et la [précision après filtrage](#exercice-3). Complétez leurs stubs puis examinez les fixtures : elles contrôlent les présences, les mots inconnus et deux mesures de précision distinctes.

## Références

- **McCallum, A. & Nigam, K. (1998).** « A Comparison of Event Models for Naive Bayes Text Classification », AAAI-98 Workshop on Learning for Text Categorization. — modèles multinomial et Bernoulli.
- **Rennie, J. D. M. (2000).** « Learning to Classify Text Documents using Word Frequencies », rapport AIM-2000-016. — utilisation des fréquences de mots pour la classification.
- **sklearn** — `MultinomialNB` / `BernoulliNB` (l'organe de référence du cross-check).

**Place dans la série** : `10` — la machine probabiliste de la série appliquée à la **classification** (après le n-gramme 02 et la recherche d'information 06).


## À retenir

- **Le classifieur est un comptage.** Estimer Naive Bayes, c'est compter : les priors `P(c)` par classe, les probabilités de mot `P(w|c)` par MLE, lisser par add-α et sommer en log-espace. C'est la tradition du carnet 02 (MLE + lissage), tournée vers les catégories — aucune boucle d'optimisation n'intervient.
- **La précision est mesurée, et l'erreur est diagnostiquable.** 7/8 hors échantillon sur le corpus embarqué. Le document mal classé n'est pas un défaut de calcul : cinq de ses huit mots sont hors vocabulaire du train, et les trois mots restants (`le`, `est`, `du`) contredisent tous `finance`. Une erreur de prédiction se lit dans **les mots réellement comptés**, pas dans les scores.
- **Le cross-check valide le code, pas le modèle.** `sklearn` rend le même compte (7/8) et le même verdict sur chacun des huit documents. Cet accord prouve que la machine from scratch calcule ce qu'elle prétend ; il ne corrige pas l'absence d'information qui produit l'erreur — elle est dans les données, et le témoin la reproduit.
- **Un plateau de précision n'est pas une insensibilité.** De α = 0,1 à α = 5, la précision reste à 7/8, mais la marge de décision médiane passe de 6,33 à 0,59 nat : la **confiance** est divisée par onze quand le compte ne bouge pas. La résolution du test — huit documents, 12,5 points chacun — masque ce que la marge, elle, montre.
- **Suite de la série** : le carnet 11 attaque le **même problème** par la voie discriminative — modéliser `P(classe | mots)` directement, sans hypothèse d'indépendance.

## Vérifiez votre compréhension

1. Le document « le dividende est verse aux actionnaires du marche » est classé `sport`. Où se lit la cause réelle de cette erreur, et pourquoi n'est-ce pas un défaut d'implémentation ?
2. L'accord avec `sklearn` porte sur les huit documents de test. Que prouve-t-il, et que ne prouve-t-il pas ?
3. La précision reste à 7/8 pour toutes les valeurs de α testées au-delà de 0,1. Que cache ce plateau, et que mesure la colonne « marge mediane » qu'il ne montre pas ?
4. À α = 0, trois documents sont dits « indéterminés » et non « mal classés ». Quelle mécanique produit ce statut, et pourquoi la colonne marge y affiche-t-elle « - » ?

<details><summary>Réponses</summary>

1. Cinq des huit mots sont hors vocabulaire du train (`dividende`, `verse`, `aux`, `actionnaires`, `marche`). Ne restent que `le`, `est`, `du`, dont les comptages contredisent `finance` — deux la contredisent, un lui est absent. Le scoreur n'a reçu aucune preuve utile : l'absence d'information a été traitée comme information neutre (mots hors vocabulaire ignorés). La cause se lit dans les comptages, avant les scores.
2. Il prouve que la machine from scratch **calcule ce qu'elle prétend calculer** — même tokenisation, même vocabulaire construit sur le train, même ignorance des mots hors vocabulaire, même estimateur. Il ne prouve rien sur le modèle : le témoin reproduit la même erreur, parce que l'absence d'information est dans les données, pas dans l'algorithme.
3. Le test n'a que huit documents : sa résolution est de 12,5 points par document, insuffisante pour distinguer des valeurs voisines de α. La marge médiane — écart de log-score entre la 1re et la 2e catégorie — passe de 6,33 nat (α = 0,1) à 0,59 nat (α = 5) : la confiance chute d'un facteur onze à précision constante. À α = 5, le terme de lissage α·|V| = 590 pèse plus de dix fois les jetons réellement observés d'une catégorie.
4. Sans lissage, un seul mot du vocabulaire absent d'une classe rend son score `-inf`, et un seul `-inf` contamine toute la somme de log-probabilités. Les trois documents où **toutes** les classes sont à `-inf` ne reçoivent aucune prédiction — c'est l'indétermination. La marge y est « - » parce qu'aucun document classé ne garde deux scores finis : la décision n'est plus un arbitrage entre catégories, mais la survie d'une seule.

</details>

## Pour aller plus loin

**Carnet suivant** : [11-Classement-Texte-RegressionLogistique-FromScratch.ipynb](11-Classement-Texte-RegressionLogistique-FromScratch.ipynb) — le même corpus socle, le même problème, mais la voie **discriminative** : modéliser directement `P(classe | mots)` sans hypothèse d'indépendance, et payer ce choix par une optimisation itérative.

- **Lectures** : Jurafsky & Martin, *Speech and Language Processing* (3e éd., draft), chapitre « Naive Bayes » — la dérivation du modèle et la discussion des modèles d'événement ; Manning, Raghavan & Schütze, *Introduction to Information Retrieval*, ch. 13 — le cadre génératif appliqué à la classification de textes.
- **Dans le dépôt** : [02_NGrammes_Modeles_De_Langue.ipynb](02_NGrammes_Modeles_De_Langue.ipynb) — la même paire MLE + lissage, côté modèle de langue ; [06-Recherche-Information-TFIDF-BM25.ipynb](06-Recherche-Information-TFIDF-BM25.ipynb) — le même sac-de-mots, côté recherche ; l'exercice 2 de ce carnet, qui demande l'explication d'une prédiction mot à mot — la lecture des comptages pratiquée en section 2.